# Telco 客戶流失：端到端資料科學專案（教學版）

本筆記本對應專案目錄：

```
專案根目錄/
├── data/raw/Telco-Customer-Churn.csv   # 原始資料
├── src/telco_churn/cleaning.py         # 可重用的清理函式
├── notebooks/本檔.ipynb
├── requirements.txt
└── README.md
```

**商業問題**：哪些客戶較可能解約（Churn）？名單可用於留客預算與客服優先序（離線建模示範，非上線管線）。

## 流程總覽

| 階段 | 說明 |
|------|------|
| 讀取 | 自 `data/raw` 載入 CSV |
| 清理 | 模組化函式 `clean_telco_churn` |
| EDA | 結構、目標分佈、視覺化、分組流失率 |
| 前處理 | One-Hot、train/test 分層切分 |
| 建模 | 邏輯迴歸、隨機森林、HistGradientBoosting |
| 評估 | ROC-AUC、classification_report |

> **📌 本節目的**：對齊「可重現的專案結構」與問題陳述。  
> **為什麼**：GitHub／教學專案需要讓讀者知道檔案從哪來、程式從哪跑，避免只見 notebook 不見資料與模組。


## 0. 環境與資料路徑

從專案根目錄或 `notebooks/` 啟動 kernel 皆可：下列程式會向上尋找含 `data/raw` 的專案根目錄，並把 `src` 加入 `sys.path` 以匯入清理模組。


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 專案根目錄（含 data/raw 與 src）
ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / "data" / "raw" / "Telco-Customer-Churn.csv").exists():
        break
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from telco_churn.cleaning import clean_telco_churn

CSV_PATH = ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
RANDOM_STATE = 42

print("專案根目錄:", ROOT)
print("資料檔:", CSV_PATH, CSV_PATH.exists())


> **📌 本格目的**：可攜帶的路徑解析與匯入專案模組。  
> **為什麼**：教學 repo 常見「在 notebooks 裡跑就找不到 `src`」；固定搜尋根目錄可減少這類錯誤。


## 1. 讀取原始資料

使用 `pandas.read_csv`；若遇編碼問題可改 `encoding='utf-8-sig'` 等。


In [ ]:
df = pd.read_csv(CSV_PATH)
print("形狀:", df.shape)
df.head()


> **📌 本格目的**：載入與 README 同一來源的檔案。  
> **為什麼**：後續指標必須能對檔名與列數追溯；也是資料契約的第一步。


## 2. 數據清理

規則實作於 `src/telco_churn/cleaning.py`（與本筆記本共用）。重點：欄名與字串 strip、主鍵去重、`TotalCharges` 數值化與填補、合法類別與數值範圍。


In [ ]:
df = clean_telco_churn(df)

print("\n清理後缺失（應無或已處理）:")
miss = df.isna().sum()
print(miss[miss > 0] if miss.sum() else "無剩餘缺失")


> **📌 本格目的**：讓每位客戶一列、型別正確。  
> **為什麼**：重複 ID 會扭曲流失率；`TotalCharges` 字串會讓模型無法使用；新戶缺總帳需符合營運解釋（年資 0）。


## 3. 探索性分析（EDA）

在已清理的 `df` 上檢視型別、描述統計、目標比例與月費與 Churn 的關係。


In [ ]:
df.info()
df.describe(include="all")


In [ ]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("\nChurn 比例:")
print(df["Churn"].value_counts(normalize=True))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["Churn"].value_counts().plot(kind="bar", ax=axes[0], title="Churn count")
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[1])
plt.suptitle("")
axes[1].set_title("MonthlyCharges vs Churn")
plt.tight_layout()
plt.show()


> **📌 本格目的**：確認不平衡與月費分佈差異。  
> **為什麼**：類別比例影響指標解讀；圖形有助形成可驗證的假說。


### 3.1 商業洞察：分組流失率

將資料翻成營運語言（合約、付款、方案、年資）；若與後續模型特徵方向一致，較易取得跨部門信任。**相關不等於因果**。


In [ ]:
def churn_rate_by(col: str) -> pd.Series:
    return df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)

print("整體流失率:", f"{(df['Churn'] == 'Yes').mean():.1%}")
print("\nContract:")
print(churn_rate_by("Contract"))
print("\nPaymentMethod:")
print(churn_rate_by("PaymentMethod"))
print("\nInternetService:")
print(churn_rate_by("InternetService"))

df["_tb"] = pd.cut(
    df["tenure"],
    bins=[-1, 0, 12, 24, 60, 1000],
    labels=["0", "1-12", "13-24", "25-60", ">60"],
)
print("\nTenure bin:")
print(df.groupby("_tb", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean()))
df.drop(columns=["_tb"], inplace=True)

p75 = df["TotalCharges"].quantile(0.75)
hq_m2m = (df["Contract"] == "Month-to-month") & (df["TotalCharges"] >= p75)
print("\n高累計(P75+)且月租 — 流失率:", f"{(df.loc[hq_m2m, 'Churn'] == 'Yes').mean():.1%}", "n=", int(hq_m2m.sum()))


> **📌 本格目的**：對齊常見電信業敘事（月租、電子支票、Fiber、年資）。  
> **為什麼**：模型需能向業務交代「為何這份名單」；分組率是最直覺的對照。


## 4. 客戶價值分析

從「月費 × 在網時間」的角度，找出高價值客戶在哪裡流失。

In [ ]:
# 高價值客戶定義：月費 > 中位數 且 在網 > 12 個月
median_charge = df["MonthlyCharges"].median()
high_value = df[(df["MonthlyCharges"] > median_charge) & (df["tenure"] > 12)].copy()

total_customers = len(df)
hv_count = len(high_value)
hv_churn = (high_value["Churn"] == "Yes").mean()
overall_churn = (df["Churn"] == "Yes").mean()

print(f"整體客戶數：{total_customers:,}")
print(f"高價值客戶數：{hv_count:,}（佔 {hv_count/total_customers:.1%}）")
print(f"整體流失率：{overall_churn:.1%}")
print(f"高價值客戶流失率：{hv_churn:.1%}")
print(f"\n➜ 高價值客戶流失率比整體{'高' if hv_churn > overall_churn else '低'} {abs(hv_churn - overall_churn):.1%}")


> **商業意涵**：高價值客戶流失代表直接的收益損失，比一般客戶更值得投入留客資源。

## 5. 流失客戶的月費損失估算

把流失行為轉換成可以向老闆報告的數字。

In [ ]:
churned = df[df["Churn"] == "Yes"]
retained = df[df["Churn"] == "No"]

monthly_loss = churned["MonthlyCharges"].sum()
avg_loss_per_customer = churned["MonthlyCharges"].mean()
avg_tenure_churned = churned["tenure"].mean()
avg_tenure_retained = retained["tenure"].mean()

print(f"流失客戶數：{len(churned):,}")
print(f"每月流失月費收入：${monthly_loss:,.0f}")
print(f"每位流失客戶平均月費：${avg_loss_per_customer:.2f}")
print(f"\n流失客戶平均在網時間：{avg_tenure_churned:.1f} 個月")
print(f"留存客戶平均在網時間：{avg_tenure_retained:.1f} 個月")
print(f"\n➜ 留存客戶在網時間比流失客戶多 {avg_tenure_retained - avg_tenure_churned:.1f} 個月")


> **商業意涵**：在網時間差距顯示，若能在客戶流失前介入，可顯著提升客戶終身價值（LTV）。

## 6. 哪些服務組合最容易留住客戶？

找出「低流失率服務組合」，提供給行銷部門作為推廣重點。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 合約類型 vs 網路服務的流失率熱圖
pivot = df.groupby(["Contract", "InternetService"])["Churn"].apply(
    lambda s: (s == "Yes").mean()
).unstack()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(pivot, annot=True, fmt=".1%", cmap="RdYlGn_r",
            linewidths=0.5, ax=axes[0])
axes[0].set_title("流失率：合約類型 × 網路服務")
axes[0].set_xlabel("網路服務")
axes[0].set_ylabel("合約類型")

# 付款方式流失率長條圖
pay_churn = df.groupby("PaymentMethod")["Churn"].apply(
    lambda s: (s == "Yes").mean()
).sort_values(ascending=True)

pay_churn.plot(kind="barh", ax=axes[1], color=["#2ecc71","#27ae60","#e67e22","#e74c3c"])
axes[1].set_title("流失率：付款方式")
axes[1].set_xlabel("流失率")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

plt.tight_layout()
plt.savefig("churn_by_service.png", dpi=150, bbox_inches="tight")
plt.show()
print("圖表已儲存")


> **商業意涵**：合約長度與付款方式是最直接的行為指標，行銷可優先推廣年約＋自動扣款組合。

## 7. 新客戶危險期分析

找出流失最集中的在網月份，協助客服設定主動關懷時間點。

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

# 按在網時間分組，計算流失率
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 3, 6, 12, 24, 48, 1000],
    labels=["1-3月", "4-6月", "7-12月", "13-24月", "25-48月", "48月以上"],
    right=True
)

tenure_churn = df.groupby("tenure_group", observed=True)["Churn"].apply(
    lambda s: (s == "Yes").mean()
)

bars = ax.bar(tenure_churn.index, tenure_churn.values,
              color=["#e74c3c" if v > 0.3 else "#e67e22" if v > 0.2 else "#2ecc71"
                     for v in tenure_churn.values])

ax.axhline(y=(df["Churn"] == "Yes").mean(), color="gray",
           linestyle="--", label=f"整體平均 {(df['Churn'] == 'Yes').mean():.1%}")
ax.set_title("不同在網時間的流失率")
ax.set_xlabel("在網時間")
ax.set_ylabel("流失率")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend()

for bar, val in zip(bars, tenure_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("churn_by_tenure.png", dpi=150, bbox_inches="tight")
plt.show()

df.drop(columns=["tenure_group"], inplace=True)


> **商業意涵**：前三個月是最高危險期，建議在客戶加入第1、3個月時主動聯繫，提供專屬優惠或使用引導。

## 8. 商業建議摘要

根據以上分析，整理三個可執行的行動方向。

In [ ]:
summary = {
    "發現": [
        "月租型客戶流失率遠高於一年期與兩年期合約",
        "使用電子支票付款的客戶流失率最高",
        "新客戶前3個月是最高風險期",
        "Fiber光纖客戶流失率高於DSL，顯示服務品質可能有落差",
    ],
    "建議行動": [
        "【合約轉換】針對月租客戶提供年約折扣，目標是把月租轉換為年約",
        "【付款升級】推動電子支票客戶改用信用卡自動扣款，降低流失風險",
        "【新客關懷】建立第1、3個月的主動聯繫機制（電話或Email）",
        "【Fiber品質】調查Fiber客戶不滿意的具體原因，優先改善",
    ]
}

print("=" * 50)
print("客戶流失分析 — 商業建議摘要")
print("=" * 50)
for category, items in summary.items():
    print(f"\n【{category}】")
    for i, item in enumerate(items, 1):
        print(f"  {i}. {item}")
print("\n" + "=" * 50)


---

## 9. 本專案定位說明

本專案以**商業洞察**為核心，聚焦在「把數據翻譯成老闆看得懂的結論」：

- ✅ 流失率分組分析（合約、付款、年資、服務）
- ✅ 月費損失金額估算
- ✅ 新客危險期識別
- ✅ 可執行的行動建議

**刻意未包含**：機器學習模型、預測分數、A/B testing——這些屬於更進階的數據科學範疇，不在本次分析範圍內。

> 資料來源：IBM Telco Customer Churn 公開資料集
